In [0]:
"""
scripts/ingest.py — Data Ingestion (Extract Phase)
School Enrollment & Education Performance Analytics Platform
"""
import os, logging
import pandas as pd
from datetime import datetime

os.makedirs("logs", exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    handlers=[logging.StreamHandler(),
              logging.FileHandler(f"logs/ingest_{datetime.now().strftime('%Y%m%d')}.log")]
)
logger = logging.getLogger(__name__)

RAW_CSV_PATH   = "/Volumes/education_analytics/default/education_analytics_platform/enrollment_data_raw.csv"
PROCESSED_PATH = "data/processed/"
REQUIRED_COLS  = ["record_id","school_name","region","school_type","academic_year",
                   "grade","subject","male_count","female_count","total_enrollment",
                   "dropout_rate","avg_performance_score","pass_rate"]

def validate_file(path):
    if not os.path.exists(path): raise FileNotFoundError(f"Not found: {path}")
    if os.path.getsize(path) == 0: raise ValueError(f"Empty file: {path}")
    logger.info(f"File OK: {path} ({os.path.getsize(path)/1024:.1f} KB)")

def load_raw(path):
    logger.info(f"Loading: {path}")
    df = pd.read_csv(path, dtype=str, keep_default_na=True)
    logger.info(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
    return df

def validate_schema(df):
    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing: raise ValueError(f"Missing columns: {missing}")
    logger.info(f"Schema OK — {len(REQUIRED_COLS)} required columns present")

def run_ingestion():
    logger.info("=== INGESTION START ===")
    try:
        validate_file(RAW_CSV_PATH)
        df = load_raw(RAW_CSV_PATH)
        validate_schema(df)
        logger.info(f"Missing cells: {df.isnull().sum().sum():,}")
        logger.info(f"Duplicate rows: {df.duplicated().sum():,}")
        os.makedirs(PROCESSED_PATH, exist_ok=True)
        out = os.path.join(PROCESSED_PATH, "bronze_raw.csv")
        df.to_csv(out, index=False)
        logger.info(f"Saved to: {out}")
        logger.info("=== INGESTION COMPLETE ===")
        return df
    except Exception as e:
        logger.error(f"FAILED: {e}"); raise

if __name__ == "__main__":
    df = run_ingestion()
    print(f"Done: {len(df):,} rows")